In [ ]:
from dataclasses import dataclass
from pathlib import Path

MODELS = {
    "qwen3tts": {"sr": 24000, "min_dur": 2.0, "max_dur": 150.0, "pad_end": 0.5},
    "cosyvoice2": {"sr": 16000, "min_dur": 2.0, "max_dur": 150.0, "pad_end": 0.0},
}

TARGET_MODEL = "qwen3tts"          # "qwen3tts" or "cosyvoice2"
AUDIO_PATH = "mic_recording.wav"        # path to your voice reference
TRANSCRIPT = "When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arch, with its path high above, and its two ends apparently beyond the horizon. There is, according to legend, a boiling pot of gold at one end. People look, but no one ever finds it. When a man looks for something beyond his reach, his friends say he is looking for the pot of gold at the end of the rainbow. Throughout history, the rainbow has been a symbol of hope and a sign of things to come. The vibrant bands of red, orange, yellow, green, blue, and violet curve gracefully across the sky, reminding us of the calm that follows a storm. Scientists observe these wavelengths to understand the physics of light, while artists simply try to capture their fleeting brilliance on canvas."
OUTPUT_DIR = Path("processed")
DENOISE = True
DENOISE_STRENGTH = 0.5           
TARGET_LUFS = -23.0
WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo-asr-fp16" # Whisper model for testing only

MODEL_CFG = MODELS[TARGET_MODEL]
print(f"Model: {TARGET_MODEL} | SR: {MODEL_CFG['sr']}Hz | Duration: {MODEL_CFG['min_dur']}-{MODEL_CFG['max_dur']}s")

Model: qwen3tts | SR: 24000Hz | Duration: 2.0-150.0s


In [2]:
import numpy as np
import soundfile as sf
import soxr
import pyloudnorm as pyln
import json
from pathlib import Path


def load_audio(path):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if audio.shape[1] > 1:
        audio = np.mean(audio, axis=1)
    else:
        audio = audio[:, 0]
    return audio, sr


def trim_silence(audio, sr, threshold_db=-40.0, pad_start_ms=50, pad_end_ms=100):
    if len(audio) == 0:
        return audio
    hop = int(0.02 * sr)
    peak = np.max(np.abs(audio))
    if peak < 1e-10:
        return audio
    thresh = peak * (10 ** (threshold_db / 20.0))
    n_frames = len(audio) // hop
    if n_frames == 0:
        return audio

    first, last = 0, n_frames - 1
    for i in range(n_frames):
        if np.max(np.abs(audio[i*hop:i*hop+hop])) > thresh:
            first = i; break
    for i in range(n_frames-1, -1, -1):
        if np.max(np.abs(audio[i*hop:i*hop+hop])) > thresh:
            last = i; break

    start = max(0, first * hop - int(pad_start_ms / 1000 * sr))
    end = min(len(audio), (last + 1) * hop + int(pad_end_ms / 1000 * sr))
    return audio[start:end]


def spectral_denoise(audio, sr, strength=0.5, n_fft=2048):
    if strength <= 0:
        return audio
    hop = n_fft // 4
    n_frames = (len(audio) - n_fft) // hop + 1
    if n_frames < 4:
        return audio
    win = np.hanning(n_fft)
    frames = np.array([audio[i*hop:i*hop+n_fft] * win for i in range(n_frames)])
    spec = np.fft.rfft(frames, axis=1)
    mag, phase = np.abs(spec), np.angle(spec)

    energies = np.mean(mag**2, axis=1)
    n_noise = max(1, int(n_frames * 0.1))
    noise_profile = np.mean(mag[np.argsort(energies)[:n_noise]], axis=0)

    clean_mag = np.maximum(mag - noise_profile * strength * 2, mag * 0.01)
    clean = np.fft.irfft(clean_mag * np.exp(1j * phase), n=n_fft, axis=1)

    out = np.zeros(len(audio), dtype=np.float32)
    wsum = np.zeros(len(audio), dtype=np.float32)
    for i in range(n_frames):
        s = i * hop
        out[s:s+n_fft] += clean[i] * win
        wsum[s:s+n_fft] += win**2
    mask = wsum > 1e-8
    out[mask] /= wsum[mask]
    return out


def normalize_loudness(audio, sr, target_lufs=-23.0):
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio)
    if np.isinf(loudness) or np.isnan(loudness):
        return audio
    return np.clip(pyln.normalize.loudness(audio, loudness, target_lufs), -1.0, 1.0)


print("Functions loaded.")

Functions loaded.


In [3]:
assert TRANSCRIPT, f"{TARGET_MODEL} requires a transcript for the reference audio."

audio, orig_sr = load_audio(AUDIO_PATH)
orig_dur = len(audio) / orig_sr
print(f"Loaded: {orig_dur:.2f}s at {orig_sr}Hz")

# Trim silence
audio = trim_silence(audio, orig_sr)
print(f"After trim: {len(audio)/orig_sr:.2f}s")

# Denoise
if DENOISE:
    audio = spectral_denoise(audio, orig_sr, strength=DENOISE_STRENGTH)
    print(f"Denoised (strength={DENOISE_STRENGTH})")

# Resample
target_sr = MODEL_CFG["sr"]
if orig_sr != target_sr:
    audio = soxr.resample(audio, orig_sr, target_sr, quality="HQ")
    print(f"Resampled: {orig_sr} -> {target_sr}Hz")

# Normalize loudness
audio = normalize_loudness(audio, target_sr, TARGET_LUFS)
print(f"Loudness normalized: {TARGET_LUFS} LUFS")

# Validate duration
duration = len(audio) / target_sr
if duration < MODEL_CFG["min_dur"]:
    raise ValueError(f"Too short: {duration:.2f}s < {MODEL_CFG['min_dur']}s. Record a longer clip.")
if duration > MODEL_CFG["max_dur"]:
    audio = audio[:int(MODEL_CFG["max_dur"] * target_sr)]
    print(f"Truncated: {duration:.2f}s -> {MODEL_CFG['max_dur']}s")

# Model-specific padding
if MODEL_CFG["pad_end"] > 0:
    pad = np.zeros(int(MODEL_CFG["pad_end"] * target_sr), dtype=np.float32)
    audio = np.concatenate([audio, pad])
    print(f"Padded end: +{MODEL_CFG['pad_end']}s (phoneme bleed fix)")

final_dur = len(audio) / target_sr
print(f"\nFinal: {final_dur:.2f}s at {target_sr}Hz")

Loaded: 65.54s at 48000Hz
After trim: 65.54s
Denoised (strength=0.5)
Resampled: 48000 -> 24000Hz
Loudness normalized: -23.0 LUFS
Padded end: +0.5s (phoneme bleed fix)

Final: 66.04s at 24000Hz


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stem = Path(AUDIO_PATH).stem
out_path = OUTPUT_DIR / f"{stem}_{TARGET_MODEL}_ref.wav"

sf.write(str(out_path), audio, target_sr, subtype="PCM_16")
print(f"Saved: {out_path}")

Saved: processed/mic_recording_qwen3tts_ref.wav


In [ ]:
from mlx_audio.stt.generate import generate_transcription

result = generate_transcription(
    model=WHISPER_MODEL,
    audio=str(out_path),
)

TRANSCRIPT = result.text.strip()
print(f"Transcript: {TRANSCRIPT}")

/Users/nicholastristan_1/Apple Institute/Challenge 1 Audio/Code/.audioenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 6603/6603 [00:01<00:00, 3828.44frames/s]

Transcript: When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arc, with its path high above and its two ends apparently beyond the horizon. There is, according to legend, a boiling pot of gold at one end. People look, but no one ever finds it. When a man looks for something beyond his reach, his friends say he is looking for the pot of gold at the end of the rainbow. Throughout history, the rainbow has been a symbol of hope and a sign of things to come. The vibrant bands of red, orange, yellow, green, blue, and violet curve gracefully across the sky, reminding us of the calm that follows a storm. Scientists observe these wavelengths to understand the physics of light, while artists simply try to capture their fleeting brilliance on canvas.


In [6]:
# --- IMPORTANT: Verify & Edit Transcript ---
# Whisper may mishear proper nouns, homophones, or filler words.
# Listen to the audio and fix any errors below before using for TTS.

# Uncomment and edit if Whisper got something wrong:
# TRANSCRIPT = "Your manually corrected transcript here."

print(f"Final transcript: {TRANSCRIPT}")

Final transcript: When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arc, with its path high above and its two ends apparently beyond the horizon. There is, according to legend, a boiling pot of gold at one end. People look, but no one ever finds it. When a man looks for something beyond his reach, his friends say he is looking for the pot of gold at the end of the rainbow. Throughout history, the rainbow has been a symbol of hope and a sign of things to come. The vibrant bands of red, orange, yellow, green, blue, and violet curve gracefully across the sky, reminding us of the calm that follows a storm. Scientists observe these wavelengths to understand the physics of light, while artists simply try to capture their fleeting brilliance on canvas.


In [7]:
meta = {
    "model": TARGET_MODEL,
    "original_sr": orig_sr,
    "target_sr": target_sr,
    "original_duration": round(orig_dur, 3),
    "final_duration": round(final_dur, 3),
    "transcript": TRANSCRIPT,
    "denoise": DENOISE,
    "lufs": TARGET_LUFS,
}
meta_path = out_path.with_suffix(".json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved: {out_path}")
print(f"Meta:  {meta_path}")

Saved: processed/mic_recording_qwen3tts_ref.wav
Meta:  processed/mic_recording_qwen3tts_ref.json
